In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [3]:
df = pd.read_csv('../data/q3_retail_promotions.csv')
print(df.head())

  transaction_date  store_id store_size location_type  promotion_type  \
0       2022-01-01        28      small    semi-urban       free_gift   
1       2022-01-01         5     medium    semi-urban       free_gift   
2       2022-01-02        13      small    semi-urban  loyalty_points   
3       2022-01-02        17      small         urban       free_gift   
4       2022-01-03        50     medium    semi-urban            bogo   

   is_weekend  is_festival  competition_density  items_sold  
0           1            0                    5         224  
1           1            1                    1         348  
2           1            0                    6         249  
3           1            0                    7         259  
4           0            0                    3         277  


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   transaction_date     1200 non-null   datetime64[us]
 1   store_id             1200 non-null   int64         
 2   store_size           1200 non-null   str           
 3   location_type        1200 non-null   str           
 4   promotion_type       1200 non-null   str           
 5   is_weekend           1200 non-null   int64         
 6   is_festival          1200 non-null   int64         
 7   competition_density  1200 non-null   int64         
 8   items_sold           1200 non-null   int64         
 9   year                 1200 non-null   int32         
 10  month                1200 non-null   int32         
 11  day_of_week          1200 non-null   str           
 12  is_month_end         1200 non-null   int64         
dtypes: datetime64[us](1), int32(2), int64(6), st

In [9]:
# 1. Date Feature Engineering
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['year'] = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month
df['day_of_week'] = df['transaction_date'].dt.day_name()
df['is_month_end'] = (df['transaction_date'].dt.day >= 25).astype(int)
df.head()

,transaction_date,store_id,store_size,location_type,promotion_type,is_weekend,is_festival,competition_density,items_sold,year,month,day_of_week,is_month_end
0,2022-01-01,28,small,semi-urban,free_gift,1,0,5,224,2022,1,Saturday,0
1,2022-01-01,5,medium,semi-urban,free_gift,1,1,1,348,2022,1,Saturday,0
2,2022-01-02,13,small,semi-urban,loyalty_points,1,0,6,249,2022,1,Sunday,0
3,2022-01-02,17,small,urban,free_gift,1,0,7,259,2022,1,Sunday,0
4,2022-01-03,50,medium,semi-urban,bogo,0,0,3,277,2022,1,Monday,0


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   transaction_date     1200 non-null   datetime64[us]
 1   store_id             1200 non-null   int64         
 2   store_size           1200 non-null   str           
 3   location_type        1200 non-null   str           
 4   promotion_type       1200 non-null   str           
 5   is_weekend           1200 non-null   int64         
 6   is_festival          1200 non-null   int64         
 7   competition_density  1200 non-null   int64         
 8   items_sold           1200 non-null   int64         
 9   year                 1200 non-null   int32         
 10  month                1200 non-null   int32         
 11  day_of_week          1200 non-null   str           
 12  is_month_end         1200 non-null   int64         
dtypes: datetime64[us](1), int32(2), int64(6), st

In [13]:
# 2. Temporal Train-Test Split
df = df.sort_values('transaction_date').reset_index(drop=True)

split = int(len(df) * 0.8)

train_df = df.iloc[:split]
test_df = df.iloc[split:]

target = 'items_sold'
features = ['year', 'month', 'day_of_week', 'is_month_end', 'store_size', 'location_type', 'promotion_type', 'is_weekend', 'is_festival', 'competition_density']

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")

Training set: (960, 10)
Testing set: (240, 10)


Random split is inappropriate for time ordered data, because spliting data randomly can shuffle the past and future dates. The model would train on future dates and test on past dates, which is not ideal. In retail, sales usually follows a pattern over the days. This pattern would be lost if we were to shuffle them.

In [14]:
# 3. Preprocessing Pipeline
categorical_cols = ['promotion_type', 'location_type', 'store_size', 'day_of_week']
numerical_cols = ['year', 'month', 'is_month_end', 'is_weekend', 'is_festival', 'competition_density']

preprocessor = make_column_transformer(
    (OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_cols),
    (StandardScaler(), numerical_cols),
    verbose_feature_names_out=False
)

In [ ]:
# 4. Model Training and Evaluation
models = {
    "Linear Regression": make_pipeline(preprocessor, LinearRegression()),
    "Random Forest": make_pipeline(preprocessor, RandomForestRegressor(n_estimators=100, random_state=42))
}

In [ ]:
# Lianer Regression
lr_model = make_pipeline(preprocessor, LinearRegression())
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)
print(f"Linear Regression RMSE: {round(rmse_lr, 2)}")
print(f"Linear Regression MAE: {round(mae_lr, 2)}")

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_lr, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--')
plt.xlabel('Actual Items Sold')
plt.ylabel('Predicted Items Sold')
plt.title('Linear Regression Parity Plot')
plt.grid(True)
plt.show()

In [ ]:
# Random Forest Regressor
rf_model = make_pipeline(preprocessor, RandomForestRegressor(n_estimators=100, random_state=42))
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
print(f"Random Forest RMSE: {round(rmse_rf, 2)}")
print(f"Random Forest MAE: {round(mae_rf, 2)}")

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_rf, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--')
plt.xlabel('Actual Items Sold')
plt.ylabel('Predicted Items Sold')
plt.title('Random Forest Parity Plot')
plt.grid(True)
plt.show()

In [ ]:
# Feature Importance
rf_step = models["Random Forest"].named_steps['randomforestregressor']
feature_names = models["Random Forest"].named_steps['columntransformer'].get_feature_names_out()

importances = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_step.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 5 Most Influential Features")
print(importances.head())